# Rule Induction

**Track:** Learning | **Function:** Procedural rule discovery

**What it measures:** Can the model discover hidden transformation rules from input/output examples?

Each item shows examples of a novel, procedurally-generated string transformation. The model must induce the rule and apply it to a new input.

**Pre/post learning format:**
- *Baseline:* Apply the rule directly from examples
- *Post-learning:* Study examples, generate analysis notes, then apply with notes as context

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import string
import re

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_short_answer(response: str) -> str:
    """Extract a short answer from model response (last short line)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Prefer last short line (skip explanations)
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

In [ ]:
# === Composable transformation operations ===

def op_caesar(s, k):
    return ''.join(chr((ord(c)-97+k)%26+97) if c.isalpha() else c for c in s)

def op_reverse(s, _):
    return s[::-1]

def op_swap_pairs(s, _):
    r = list(s)
    for i in range(0, len(r)-1, 2):
        r[i], r[i+1] = r[i+1], r[i]
    return ''.join(r)

def op_vowel_shift(s, _):
    m = {'a':'e','e':'i','i':'o','o':'u','u':'a'}
    return ''.join(m.get(c,c) for c in s)

def op_rotate_half(s, _):
    mid = len(s)//2
    return s[mid:] + s[:mid]

def op_consonant_shift(s, k):
    cons = 'bcdfghjklmnpqrstvwxyz'
    return ''.join(cons[(cons.index(c)+k)%len(cons)] if c in cons else c for c in s)

OPERATIONS = [
    ('caesar', op_caesar), ('reverse', op_reverse),
    ('swap_pairs', op_swap_pairs), ('vowel_shift', op_vowel_shift),
    ('rotate_half', op_rotate_half), ('consonant_shift', op_consonant_shift),
]

WORDS = [
    'planet','bridge','castle','forest','garden','hunter','jungle','knight',
    'lemon','market','orange','purple','silver','tunnel','winter','broken',
    'candle','desert','falcon','gentle','harbor','insect','jumble','kitten',
    'luster','mangle','nickel','oyster','pencil','quiver','rattle','simple',
    'temple','velvet','wander','basket','coffee','donkey',
]

def generate_rule(seed, difficulty):
    rng = random.Random(seed)
    n = {'easy':1,'medium':2,'hard':3}[difficulty]
    chosen = rng.sample(OPERATIONS, n)
    params = [rng.randint(1,5) for _ in range(n)]
    return [(name, fn, p) for (name, fn), p in zip(chosen, params)]

def apply_rule(word, rule):
    r = word.lower()
    for _, fn, p in rule:
        r = fn(r, p)
    return r

# Sanity check
r = generate_rule(42, 'medium')
print(f'Rule: {[(n,p) for n,_,p in r]}')
print(f'  planet -> {apply_rule("planet", r)}')

In [ ]:
N_EX = {'easy':6, 'medium':4, 'hard':2}
SEEDS = 15
TESTS_PER = 2
rows = []
tid = 0

for diff in ['easy','medium','hard']:
    n_ex = N_EX[diff]
    for seed in range(SEEDS):
        rule = generate_rule(seed*100 + hash(diff)%1000, diff)
        rng = random.Random(seed*7+13)
        words = rng.sample(WORDS, n_ex + TESTS_PER)
        ex_words, test_words = words[:n_ex], words[n_ex:]
        ex_text = '\n'.join(f'"{w}" -> "{apply_rule(w, rule)}"' for w in ex_words)
        for tw in test_words:
            rows.append({'task_id':tid, 'seed':seed, 'difficulty':diff,
                         'n_examples':n_ex, 'examples_text':ex_text,
                         'test_word':tw, 'expected':apply_rule(tw, rule)})
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset['difficulty'].value_counts().to_string())

In [ ]:
BASELINE_P = '''I will show you examples of a secret transformation rule applied to words.
Study the examples, then apply the SAME rule to the test word.

Examples:
{examples}

Apply the same transformation to: "{test_word}"

Reply with ONLY the transformed word. No explanation, no quotes, just the word.'''

STUDY_P = '''I will show you examples of a secret transformation rule applied to words.
Analyze these examples and figure out the EXACT rule.

Examples:
{examples}

Write a detailed analysis of the transformation rule.
Describe EACH step. Test your theory against EVERY example.'''

APPLY_P = '''You previously analyzed a transformation rule and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original examples:
{examples}

Apply the rule to: "{test_word}"

Reply with ONLY the transformed word. No explanation, no quotes, just the word.'''

## Task 1: Baseline (direct inference from examples)

In [ ]:
@kbench.task(name='rule_induction_baseline',
             description='Apply a novel transformation rule from examples alone')
def rule_induction_baseline(llm, examples_text:str, test_word:str,
                            expected:str, difficulty:str, seed:int,
                            n_examples:int) -> bool:
    resp = llm.prompt(BASELINE_P.format(examples=examples_text, test_word=test_word))
    return extract_short_answer(resp) == expected

baseline_runs = rule_induction_baseline.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=2, timeout=60, max_attempts=2)
baseline_df = baseline_runs.as_dataframe()
print(f'Baseline done. Accuracy: {baseline_df["result"].mean():.1%}' if 'result' in baseline_df.columns else 'Baseline done.')

## Task 2: Post-learning (study + generate notes + apply)

In [ ]:
@kbench.task(name='rule_induction_learning',
             description='Study examples, generate learning notes, then apply rule with notes')
def rule_induction_learning(llm, examples_text:str, test_word:str,
                            expected:str, difficulty:str, seed:int,
                            n_examples:int) -> bool:
    notes = strip_thinking(llm.prompt(STUDY_P.format(examples=examples_text)))
    resp = llm.prompt(APPLY_P.format(notes=notes, examples=examples_text, test_word=test_word))
    return extract_short_answer(resp) == expected

learning_runs = rule_induction_learning.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=2, timeout=120, max_attempts=2)
learning_df = learning_runs.as_dataframe()
print(f'Learning done. Accuracy: {learning_df["result"].mean():.1%}' if 'result' in learning_df.columns else 'Learning done.')

## Results

In [ ]:
# === Combine results ===
analysis = dataset.copy()

def safe_map_results(df, col_name):
    if 'result' in df.columns:
        result_map = dict(zip(df.index if 'task_id' not in df.columns else df['task_id'], df['result']))
        analysis[col_name] = analysis['task_id'].map(result_map).fillna(False).astype(bool)
    else:
        analysis[col_name] = False

safe_map_results(baseline_df, 'baseline_correct')
safe_map_results(learning_df, 'learning_correct')

baseline_acc = analysis['baseline_correct'].mean()
learning_acc = analysis['learning_correct'].mean()
learning_gain = learning_acc - baseline_acc
room = 1.0 - baseline_acc
efficiency = learning_gain / room if room > 0 else 0.0

print('=' * 60)
print('RULE INDUCTION — LEARNING RESULTS')
print('=' * 60)
print(f'Baseline accuracy:      {baseline_acc:.1%}')
print(f'Post-learning accuracy: {learning_acc:.1%}')
print(f'Learning gain:          {learning_gain:+.1%}')
print(f'Learning efficiency:    {efficiency:.1%} (of possible improvement)')
print()

print('By Difficulty:')
print('-' * 60)
for diff in ['easy', 'medium', 'hard']:
    subset = analysis[analysis['difficulty'] == diff]
    if len(subset) == 0:
        continue
    b = subset['baseline_correct'].mean()
    l = subset['learning_correct'].mean()
    g = l - b
    print(f'  {diff:8s}  baseline={b:.1%}  learned={l:.1%}  gain={g:+.1%}  (n={len(subset)})')

print()
helped = analysis[~analysis['baseline_correct'] & analysis['learning_correct']]
hurt = analysis[analysis['baseline_correct'] & ~analysis['learning_correct']]
print(f'Learning helped (wrong->right): {len(helped)}')
print(f'Learning hurt (right->wrong):   {len(hurt)}')
print(f'Net learning impact:            {len(helped) - len(hurt):+d} items')


In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

difficulties = ['easy', 'medium', 'hard']
baseline_scores = [analysis[analysis['difficulty'] == d]['baseline_correct'].mean() for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]
learning_scores = [analysis[analysis['difficulty'] == d]['learning_correct'].mean() for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]
diff_labels = [d for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]

x = range(len(diff_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar([i - width/2 for i in x], baseline_scores, width, label='Baseline', color='#4C72B0')
ax1.bar([i + width/2 for i in x], learning_scores, width, label='Post-Learning', color='#55A868')
ax1.set_xlabel('Difficulty')
ax1.set_ylabel('Accuracy')
ax1.set_title('Baseline vs Post-Learning Accuracy')
ax1.set_xticks(list(x))
ax1.set_xticklabels(diff_labels)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [l - b for b, l in zip(baseline_scores, learning_scores)]
colors = ['#55A868' if g >= 0 else '#C44E52' for g in gains]
ax2.bar(diff_labels, gains, color=colors)
ax2.set_xlabel('Difficulty')
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('rule_induction_results.png', dpi=150, bbox_inches='tight')
plt.show()
